# Basic Monte-Carlo.

In [1]:
import numpy as np
import random
import plotly.graph_objects as go

from scipy.stats import poisson, bernoulli, randint, expon, uniform

[Statistics, Fall, 2014] On the table you have three things: fair coin, fair dice and iPhone. At the beginning the position of the coin is Head, of the dice is 1. On your iPhone is working WhatsApp where you receive on average 1 message in 15 minutes. You
are flipping coin or rolling a die each time you receive a message. On odd messages you turn coin, on even messages you
turn dice.<br>
Find the probability that after 30 minutes the position of the coin and dice is still {Head, 1}.

In [2]:
poisson.rvs(2)

1

In [3]:
# bernoulli.rvs(0.5, size = 2)

In [4]:
def experiment():
    messages = poisson.rvs(2)
    if messages == 0:
        return 1
    if messages == 1:
#         coinflip: 1=H, 0=T
        coin = bernoulli.rvs(0.5)
        return coin
    else:
        if messages%2 == 0:
            n = messages//2
            coins = bernoulli.rvs(0.5, size = n)
            dice = randint.rvs(1, 7, size = n)
        else:
            n = messages//2
            coins = bernoulli.rvs(0.5, size = n + 1)
            dice = randint.rvs(1, 7, size = n)
        
        return int(coins[-1] == 1 and dice[-1] == 1)

In [5]:
N = 50_000
results = [experiment() for i in range(N)]
ans = sum(results)/N

theor_ans = 2*np.exp(-2)+(1-3*np.exp(-2))/12
print(ans, theor_ans, abs(ans-theor_ans))

0.32114 0.32017007899740557 0.0009699210025944094


**Expectations**

[Finmat, Fall, 2023-24] Consider the market with stochastic interest rate r which is randomly chosen value in
(0, 1). In other words, $r \sim U(0, 1)$. Compounding is continuous. Find the fair price of the following
contract at this market: holder of the contract waits for some random time $\tau$ (which is Exponentially
distributed with parameter $\lambda=0.25$) and gets 1\\$ at the end. Assume that time $\tau$ is independent of rate $r$.

In [6]:
def pv():
    tau = expon.rvs(scale = 1/0.25)
    r = uniform.rvs(0,1)
    return np.exp(-tau*r)

In [7]:
N = 50_000
ans = np.mean([pv() for i in range(N)])

theor_ans = 0.25*np.log(1+1/0.25)
print(ans, theor_ans, abs(ans-theor_ans))

0.40338852810276293 0.40235947810852507 0.0010290499942378628


[Statistics, Fall, 2018] You are pizza deliveryman. You deliver pizza to hotel customers. There is a checkpoint at the entrance to the hotel area. Normally the document check takes 5 minutes, but with probability $1/2$ you are under detailed check, and then check takes $Y$ minutes where $Y$ is uniformly distributed random variable at interval $(5, 10)$. Your penalty for the delay of the delivery is $Z=\frac{1}{10}e^{X/2}$ where $X$ is your time spent at checkpoint. Find $E(Z)$.

In [8]:
def experiment():
    delay = bernoulli.rvs(0.5)
    
    if delay == 0:
        return np.exp(5 / 2) / 10
    else:
        y = uniform.rvs(5, 5)
        return np.exp(y / 2) / 10

In [9]:
N = 10_000
ans = np.mean([experiment() for i in range(N)])
ans

theor_ans = (np.exp(2.5) / 2 + (np.exp(5) - np.exp(2.5)) / 5) / 10
print(ans, theor_ans)

3.3552005182673157 3.3337380008726356


In [10]:
N = 10_000
Y = uniform.rvs(loc=0, scale=5, size=N)
X = 5 + bernoulli.rvs(0.5, size=N)*Y
np.mean(1/10*np.exp(X/2))

3.2681072395248036

# Discrete time models

### Binomial model

Simulate price $(S_k)_{k\ge 0}$ at Binomial model up to time $n$

In [11]:
t = [0, 1, 2, 3]
S0 = 100
u = 1.2
d = 0.9
p = 0.51
xs = random.choices([u, d], weights = [p, 1-p], k = 3)
prices = [S0, S0*xs[0], S0*xs[0]*xs[1], S0*xs[0]*xs[1]*xs[2]]
print(prices)

prices_modified = S0*np.cumprod([1] + xs)
print(prices_modified)

[100, 90.0, 81.0, 72.9]
[100.   90.   81.   72.9]


In [12]:
def binom_simulations(S0, u, d, p, max_time, n_paths):  
    paths = []
    for i  in range(n_paths):    
        xs = random.choices([u, d], weights = [p, 1-p], k = max_time)
        path = S0*np.cumprod([1] + xs)
        paths.append(path)
    return np.array(paths)    

In [14]:
S0 = 100
u = 1.01
d = 0.99
p = 0.51
n = 45
N = 3

t = np.arange(n + 1)
paths = binom_simulations(S0, u, d, p, n, N)
# paths

plot = go.Figure()
for path in paths:
    graph = go.Scatter(x = t, y = path)
    plot.add_trace(graph)
plot.show()

Verify distribution of $S_n$. In particular, show that sample mean close to theoretical one.

In [ ]:
S0 = 100
u = 1.01
d = 0.99
p = 0.51
n = 15
N = 200_000

paths = binom_simulations(S0, u, d, p, n, N)
Sn = np.array([path[-1] for path in paths])
# Sn > 105
prob = np.sum(Sn > 105)/N
prob

sample_mean = np.mean(Sn)
theor_mean = S0*(u*p+d*(1-p))**n
print(sample_mean, theor_mean, abs(sample_mean-theor_mean))

Error

In [ ]:
Ns = list(range(10_000, 200_001, 10_000))
errors = []
for N in Ns:
    paths = binom_simulations(S0, u, d, p, n, N)
    Sn = np.array([path[-1] for path in paths])
    sample_mean = np.mean(Sn)
    errors.append(abs(sample_mean-theor_mean))
    
plot = go.Figure()
graph = go.Scatter(x = Ns, y = errors)
plot.add_trace(graph)
plot.show()